# ⏱️ 08 - Timing and Performance

In data science, performance matters. Code that's instant on 100 rows can take minutes on 10 million. Jupyter/IPython give you tools to **measure** how long code takes, find **where** the time goes, and check how much **memory** your data uses.

## ✅ Learning Goals
- Use `%time` and `%%time` for a quick, one-time measurement
- Use `%timeit` and `%%timeit` for reliable comparisons, and read their output
- Compare two approaches **fairly**: same input, same result, setup excluded
- Compare Python loops with NumPy **vectorized** operations
- Spot repeated work, and see how runtime grows with data size
- Find the slow part of a function with `%prun` (and `%lprun`)
- Measure memory with `nbytes` and `memory_usage`, and load data efficiently with Pandas

> **The core habit:** measure first, improve the bottleneck, then measure again.

## Setup
All imports go in the first cell (notebook 02). This notebook creates one data file, in `08-OutputFiles`.

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path('08-OutputFiles')
OUT.mkdir(exist_ok=True)

## 1. Why Measure?
Performance work should start with **evidence**, not a hunch:

```text
Feels slow → measure → find the bottleneck → improve it → measure again
```

Don't optimize code just because it **looks** slow. A loop may be fine for 100 values and painful for 10 million, and a clever-looking rewrite may not help the real workload at all.

Useful questions:
- Which cell takes the longest? Which function inside it?
- Does the runtime grow too quickly as the data grows?
- Is the code doing the **same work** over and over?
- Could NumPy or Pandas do the operation **in bulk** instead of one item at a time?

## 2. `%time`: One Quick Measurement
`%time` runs a statement **once** and reports how long it took. It works on a line with an assignment, too.

In [ ]:
numbers = list(range(1_000_000))

%time total = sum(numbers)

The output shows two kinds of time:
- **Wall time** is real "clock on the wall" time, how long **you** waited. This is usually the number you care about.
- **CPU times** are how long the processor was actually busy. Wall time is much bigger than CPU time when the code is **waiting**, for example on a file, a website, or a slow disk.

✅ **Your Turn**: Use `%time` to measure summing one million integers with `np.sum(np.arange(1_000_000))`. Run the cell three times. Do you get the exact same time each run?

In [ ]:
# Your turn here


## 3. `%timeit`: Reliable Measurements
A single timing can be thrown off by other programs, background work, or one-time setup costs. `%timeit` runs an expression **many times** and reports the average. That's much more reliable for comparing two approaches.

In [ ]:
numbers = list(range(1_000))
%timeit [x**2 for x in numbers]

### Reading the Output
```text
41.2 μs ± 1.05 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
```
- **10,000 loops each**: in each run, the code was repeated 10,000 times. `%timeit` picks this count automatically, so fast code gets more loops.
- **7 runs**: that was done 7 separate times.
- **41.2 μs per loop**: on average, **one** execution took 41.2 microseconds.
- **± 1.05 μs**: the standard deviation, which shows how much the runs varied. Small is good.

| Unit | Meaning |
|---|---|
| s | second |
| ms | millisecond = 1/1,000 s |
| μs (or µs) | microsecond = 1/1,000,000 s |
| ns | nanosecond = 1/1,000,000,000 s |

### Saving the Result with `-o`
`-o` returns the timing as an object, so you can do math with it. For example, you can work out **how many times faster** one approach is:

In [ ]:
t_comp = %timeit -o [n * 2 for n in range(10_000)]
t_map = %timeit -o list(map(lambda n: n * 2, range(10_000)))

print(f'The comprehension is {t_map.average / t_comp.average:.2f}x as fast as map + lambda')

💡 For slow code, control the repetitions yourself: `%timeit -n 10 -r 3 ...` means 10 loops per run, 3 runs.

✅ **Your Turn**: Use `%timeit -o` to compare `sorted(data)` with `data.sort()` on a fresh copy, using `data = list(range(100_000, 0, -1))`. *Hint:* `data.sort()` changes the list in place, so time `data.copy().sort()` and `sorted(data.copy())` to make the comparison fair.

In [ ]:
# Your turn here


## 4. `%%time` and `%%timeit` for a Whole Cell
Put `%%time` or `%%timeit` as the **first line** of a cell to measure everything in it, not just one expression.

| Magic | Times... | Runs it... |
|---|---|---|
| `%time` | One line | Once |
| `%%time` | The whole cell | Once |
| `%timeit` | One line | Many times |
| `%%timeit` | The whole cell | Many times |

In [ ]:
%%timeit
total = 0
for i in range(1_000_000):
    total += i

⚠️ **Gotcha:** variables created inside a `%%timeit` cell **don't exist afterwards**. The cell ran in a temporary space, many times over. `%%time` runs your code normally, so its variables stay:

In [ ]:
%%time
squares = [value ** 2 for value in range(100_000)]

In [ ]:
len(squares)   # works, because %%time keeps variables. After %%timeit this would be a NameError

## 5. Comparing Fairly
A timing comparison is only meaningful when both versions:
- use the **same input** (and the same size),
- do the **same work**, with no extra setup hiding in one of them,
- produce the **same result** (check it!),
- and are measured over **multiple runs** (`%timeit`).

**Unfair:** suppose your data starts as a list. The first line below also **converts** the list to an array every time, and the second doesn't. The difference is huge, and it has nothing to do with squaring:

In [ ]:
data_list = list(range(1_000_000))
data_array = np.array(data_list)

%timeit np.array(data_list) ** 2    # converts the list AND squares it
%timeit data_array ** 2             # only squares it

**Fair:** create the input **once**, outside the timing, then time only the operation you care about. And before comparing speed, confirm that both versions give the same answer:

In [ ]:
values_list = list(range(1_000_000))
values_array = np.arange(1_000_000)

loop_result = [v ** 2 for v in values_list]
numpy_result = values_array ** 2

np.array_equal(np.array(loop_result), numpy_result)

✅ **Your Turn**: Time `sum(range(10_000))` against `np.sum(np.arange(10_000))`. Then time `np.sum(arr)` where `arr = np.arange(10_000)` was created **beforehand**. How much of NumPy's time was spent just creating the array? In a comment, explain which comparison is fair for which question.

In [ ]:
# Your turn here


## 6. Python Loops vs. NumPy
This is the payoff. NumPy (notebook 07) does the same calculation **in bulk**:
- **Python loop:** Python handles one value at a time, checking its type and looking up `**` on every pass.
- **NumPy (vectorized):** Python makes **one** call, and NumPy runs the whole loop in fast, compiled code.

### Squaring

In [ ]:
numbers_list = list(range(1_000_000))
numbers_array = np.arange(1_000_000)

def square_with_loop(values):
    result = []
    for value in values:
        result.append(value ** 2)
    return result

def square_with_numpy(values):
    return values ** 2

t_loop = %timeit -o square_with_loop(numbers_list)
t_numpy = %timeit -o square_with_numpy(numbers_array)
print(f'NumPy is about {t_loop.average / t_numpy.average:.0f}x faster')

### Filtering
A **boolean mask** (`values[values >= 700_000]`, from notebook 07) replaces a loop with an `if` inside:

In [ ]:
def passing_loop(values):
    result = []
    for score in values:
        if score >= 700_000:
            result.append(score)
    return result

def passing_numpy(values):
    return values[values >= 700_000]

%timeit passing_loop(numbers_list)
%timeit passing_numpy(numbers_array)

### Summary Statistics

In [ ]:
def mean_with_loop(values):
    total = 0
    for value in values:
        total += value
    return total / len(values)

%timeit mean_with_loop(numbers_list)
%timeit np.mean(numbers_array)

Speed isn't the only win. `values[values >= 700_000]` is also **shorter and clearer** than six lines of loop. It says *what* you want rather than *how* to do it.

⚠️ Looping **over a NumPy array** with a plain `for` loop is even slower than looping over a list, because each element is converted to a Python object one at a time. Arrays are fast only when NumPy does the looping.

✅ **Your Turn**:
1. Write a loop that adds 10 to every value in `numbers_list`, and a NumPy version for `numbers_array`. Check that the results match, then time both.
2. Do the same for selecting the values **greater than 500,000**.

In [ ]:
# Your turn here


## 7. Repeated Work and How Runtime Grows
A very common slowdown is recalculating the **same thing inside a loop**. Here, every score is divided by the top score, but `max(scores)` is recomputed on **every** pass:

In [ ]:
def normalize_slow(scores):
    return [s / max(scores) for s in scores]   # max() scans the whole list, every time

def normalize_fast(scores):
    top = max(scores)                          # compute it once
    return [s / top for s in scores]

scores = list(np.random.randint(0, 100, size=2_000))
print(normalize_slow(scores) == normalize_fast(scores))   # same result?

%timeit normalize_slow(scores)
%timeit normalize_fast(scores)

### Watch How It Scales
Now **double** the data a few times. `normalize_fast` takes about **2x** as long each time (it's *linear*). `normalize_slow` takes about **4x** as long, because it does *n* passes of *n* steps each (it's *quadratic*). On large data, that difference is everything:

In [ ]:
for n in [1_000, 2_000, 4_000]:
    data = list(range(1, n + 1))
    t_slow = %timeit -o -q -n 3 -r 3 normalize_slow(data)
    t_fast = %timeit -o -q -n 3 -r 3 normalize_fast(data)
    print(f'n={n:>5}:  slow {t_slow.average * 1000:8.2f} ms   fast {t_fast.average * 1000:6.3f} ms')

(`-q` keeps `%timeit` quiet, so only the `print` shows.)

Ask yourself: *Am I recalculating the same max, filter, conversion, or **file read** inside a loop?* If so, do it once, before the loop.

✅ **Your Turn**: Write a loop that recomputes something it doesn't need to, such as `len(my_list)` or `sum(my_list)` inside every pass. Move it outside the loop, and use `%timeit` to compare the two versions.

In [ ]:
# Your turn here


✅ **Your Turn**: Try squaring numbers with a Python loop, a list comprehension, and NumPy at several sizes (say 1,000, 100,000, and 1,000,000). How does the **gap** between them change as the size grows?

In [ ]:
# Your turn here


## 8. Finding the Slow Part: Profiling
`%time` and `%timeit` tell you how long a whole block takes. A **profiler** tells you **where** inside it the time goes. `%prun` runs a statement and reports the time spent in each function it called.

The pipeline below calls two helper functions for every value. Which one is the bottleneck?

In [ ]:
def double(value):
    return value * 2

def is_banned(value, banned):
    return value in banned          # searching a list checks items one by one

def pipeline(values, banned):
    kept = []
    for v in values:
        if not is_banned(v, banned):
            kept.append(double(v))
    return kept

banned_list = list(range(0, 20_000, 7))
%prun -s cumulative -l 6 pipeline(range(20_000), banned_list)

How to read it:
- **ncalls**: how many times the function was called
- **tottime**: time spent **inside** that function itself
- **cumtime**: time inside it **plus** everything it called

`is_banned` dominates. Searching a **list** with `in` checks items one at a time, but a **set** finds an item almost instantly. That's a one-word fix:

In [ ]:
banned_set = set(banned_list)

print(pipeline(range(20_000), banned_list) == pipeline(range(20_000), banned_set))
%timeit pipeline(range(20_000), banned_list)
%timeit pipeline(range(20_000), banned_set)

(`-s cumulative` sorts by cumulative time and `-l 6` shows only the top 6 rows. In VS Code, the report appears in the cell's output.)

Profile only code you've **already measured** to be slow. Profiling every two-line example is like using a metal detector to look for a paperclip.

### Line by Line (Optional)
When `%prun` points at one function but not at the slow **line**, the `line_profiler` package can time each line separately. It isn't part of the course requirements, so install it first with `%pip` (notebook 01):

In [ ]:
%pip install line_profiler

In [ ]:
%load_ext line_profiler
%lprun -f pipeline pipeline(range(20_000), banned_list)

The `% Time` column shows which line is the bottleneck. If you can't install `line_profiler`, split the function into pieces and `%timeit` each one.

✅ **Your Turn**: Write a function that runs **two** loops, one cheap (for example, summing a list) and one expensive (for example, checking whether each of 5,000 numbers is `in` a 5,000-item list). Use `%prun` to show which part takes the most time.

In [ ]:
# Your turn here


## 9. Measuring Memory
Fast code can still be a problem if it uses too much memory. NumPy arrays tell you exactly how much space their data takes:

In [ ]:
values = np.arange(1_000_000)

print('shape:', values.shape)
print('dtype:', values.dtype)
print('bytes:', f'{values.nbytes:,}')    # 1,000,000 values x 8 bytes each

The **dtype** decides the size of each value. If your numbers fit in a smaller type, converting with `.astype()` cuts memory, but check the range first! `int8` can only hold -128 to 127.

In [ ]:
for dtype in ['int64', 'int32', 'int16', 'float32']:
    print(f'{dtype:8} {values.astype(dtype).nbytes:>10,} bytes')

### Lists Are Bigger Than They Look
`sys.getsizeof()` reports an object's size, but for a list it counts only the list's internal "pointers", **not** the numbers themselves. Treat it as a starting point, not a full memory audit:

In [ ]:
as_list = list(range(1_000_000))

print(f'list (pointers only):    {sys.getsizeof(as_list):>12,} bytes')
print(f'list (with the ints):    {sys.getsizeof(as_list) + sum(sys.getsizeof(n) for n in as_list):>12,} bytes')
print(f'NumPy array (int64):     {values.nbytes:>12,} bytes')

✅ **Your Turn**: Create a NumPy array of one million random **floats** with `np.random.random(1_000_000)`. Check its `shape`, `dtype`, and `nbytes`. Then convert it to `float32` and compare the memory. Print the first 3 values of each. What did you give up for the smaller size?

In [ ]:
# Your turn here


## 10. Working with Data Efficiently
The same ideas apply to Pandas (notebook 06). First, let's make a realistic-sized CSV file of 100,000 sales records:

In [ ]:
np.random.seed(3603)
n = 100_000
sales = pd.DataFrame({
    'date': pd.date_range('2026-01-01', periods=n, freq='min').strftime('%Y-%m-%d'),
    'department': np.random.choice(['Books', 'Electronics', 'Home', 'Sports'], size=n),
    'store_id': np.random.randint(1, 50, size=n),
    'units': np.random.randint(1, 10, size=n),
    'amount': np.random.uniform(5, 500, size=n).round(2),
    'notes': np.random.choice(['', 'gift', 'returned', 'online order'], size=n),
})
sales.to_csv(OUT / 'sales.csv', index=False)
sales.info(memory_usage='deep')

💡 `memory_usage='deep'` counts the real size of text columns. Without it, Pandas only estimates them.

### Read Only the Columns You Need
`usecols` (notebook 06) skips the other columns entirely, which saves both time and memory:

In [ ]:
%time full = pd.read_csv(OUT / 'sales.csv')
%time slim = pd.read_csv(OUT / 'sales.csv', usecols=['department', 'amount'])

print(f'full: {full.memory_usage(deep=True).sum():>12,} bytes')
print(f'slim: {slim.memory_usage(deep=True).sum():>12,} bytes')

### Use `category` for Repeated Text
A text column with only a few distinct values, like `department`, can be stored as a **category**. Each distinct value is stored once, and every row just holds a small code:

In [ ]:
as_text = pd.read_csv(OUT / 'sales.csv', usecols=['department'])
as_category = pd.read_csv(OUT / 'sales.csv', usecols=['department'], dtype={'department': 'category'})

print(f'text:     {as_text.memory_usage(deep=True).sum():>10,} bytes')
print(f'category: {as_category.memory_usage(deep=True).sum():>10,} bytes')

### Process Huge Files in Chunks
If a file is too big to fit in memory, `chunksize` reads it a piece at a time, so you can keep a running total:

In [ ]:
total_revenue = 0
for chunk in pd.read_csv(OUT / 'sales.csv', chunksize=25_000):
    total_revenue += chunk['amount'].sum()

print(f'Total revenue: ${total_revenue:,.2f}')
print('Check:         $' + f"{full['amount'].sum():,.2f}")

### Avoid Row-by-Row Loops on DataFrames
`.iterrows()` loops over a DataFrame one row at a time, which is the Pandas version of the slow Python loop. Whole-column operations are vectorized, just like NumPy:

In [ ]:
%%time
discounted = []
for _, row in full.iterrows():
    discounted.append(row['amount'] * 0.90)

In [ ]:
%%time
full['discounted'] = full['amount'] * 0.90

✅ **Your Turn**:
1. Read `sales.csv` with `usecols` for just `store_id` and `units`. Compare its time and memory to `full`.
2. Use `.iterrows()` to compute `units * amount` for every row (on `full.head(10_000)`, to keep it quick), then do the same thing with one column operation. Time both and check that the results match.

In [ ]:
# Your turn here


## 11. Trustworthy Timings
Notebook timings can mislead when cells depend on leftover variables, or when something was **cached** by an earlier run. For results you'll report:
1. **Restart** the kernel (notebook 02).
2. Run the **setup** cells.
3. Run the **timing** cell, then repeat the measurement.
4. Record the **typical** result, not the luckiest one.

Also, keep setup **out** of the timed line. `%timeit np.sqrt(values)` measures the square root, not the creation of `values`.

✅ **Your Turn**: Restart the kernel, re-run the Setup cell and the squaring comparison in section 6, and compare the new speedup to your first run. In the cell below, explain why restarting makes results more trustworthy, and why the two numbers aren't identical.

*Your explanation here.*

---
### Summary
- **Measure, don't guess.** Find the bottleneck, improve it, then measure again.
- `%time`/`%%time` give a quick single measurement (wall time is what you waited). `%timeit`/`%%timeit` run many times for reliable comparisons. Use `-o` to save the result and compute a speedup.
- Variables created inside `%%timeit` don't survive. `%%time` keeps them.
- Compare fairly: same input, same work, same result, setup excluded.
- NumPy and Pandas **vectorized** operations beat Python loops on large data, and are often clearer too. Never loop over an array or use `.iterrows()` when a whole-column operation exists.
- Move repeated work out of loops. Watch for runtime that grows faster than the data.
- `%prun` finds the slow function, and `%lprun` finds the slow line. Lists search slowly and sets search fast.
- Check memory with `.nbytes`, `.astype()`, and `memory_usage(deep=True)`. Load less data with `usecols`, `category`, and `chunksize`.

## 🏋️ Practice

### Practice 1 — Time a Sort (easy)
Create a list of 100,000 random numbers (hint: `random.sample(range(1_000_000), 100_000)` after `import random`) and use `%timeit` to measure how long `sorted(...)` takes on it.

In [ ]:
import random
# Your code here


### Practice 2 — Loop vs. Comprehension (medium)
Using the `numbers` list below, compare `%timeit` for a `for` loop that builds a squared list with `.append()` vs. a list comprehension that does the same thing. Check that both give the same list, and use `-o` to print how many times faster the winner is.

In [ ]:
numbers = list(range(1_000))
# Time the for-loop version, then the comprehension version


### Practice 3 — Three-Way Comparison (harder)
Compare `%timeit` for squaring 100,000 numbers three ways: a `for` loop over `list(range(100_000))`, a list comprehension over the same list, and NumPy on `np.arange(100_000)`. Which is fastest? Which is slowest? Then repeat the loop version over the **NumPy array** instead of the list, and explain the result (see section 6).

In [ ]:
# Your three timing comparisons here


## 🔥 Challenge (Optional) — Where's the Line?
For very small inputs, a plain Python loop can actually be just as fast as (or faster than) NumPy, because NumPy has a small fixed overhead per call. Find roughly how small an array has to be before the loop wins, by timing both approaches at a few different sizes (e.g. 10, 100, 1,000, 100,000).

In [ ]:
# Your investigation here


## 📦 Check Your Output Files
This notebook's only output file is `sales.csv`, the generated practice data. Upload `08-OutputFiles` along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path, f'({path.stat().st_size:,} bytes)')